In [1]:
import math
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np


class NumericalIntegrationTask:
   
    def __init__(self):
        self.x0 = -2.0
        self.x1 = 2.0
        self.h1 = 1.0
        self.h2 = 0.5

    def f(self, x):
        """f(x)"""
        return (x ** 2) / (x ** 3 - 27.0)

    def F(self, x):
        """Первообразная F(x) = (1/3) * ln|x^3 - 27|"""

        return (1.0 / 3.0) * math.log(abs(x ** 3 - 27.0))

    def exact_integral(self):
        """Aналитическое значение"""

        return self.F(self.x1) - self.F(self.x0)

    def d1f(self, x):
        """Первая производная"""

        return -x * (x ** 3 + 54.0) / ((x ** 3 - 27.0) ** 2)

    def d2f(self, x):
        """Вторая производная"""

        return 2.0 * (x ** 6 + 162.0 * (x ** 3) - 729.0) / ((x ** 3 - 27.0) ** 3)

    def d4f(self, x):
        """Четвертая производная"""

        u = x ** 3
        num = 24.0 * x * (u ** 3 + 810.0 * (u ** 2) + 29160.0 * u - 196830.0)
        den = (u - 27.0) ** 5
        return num / den



def integrate_rect_left(f, a, b, h):
    """Метод левых прямоугольников (p = 1)"""

    n = int(round((b - a) / h))
    return h * sum(f(a + i * h) for i in range(n))


def integrate_rect_right(f, a, b, h):
    """Метод правых прямоугольников (p = 1)"""

    n = int(round((b - a) / h))
    return h * sum(f(a + (i + 1) * h) for i in range(n))


def integrate_rect_mid(f, a, b, h):
    """Метод средних прямоугольников (p = 2)"""

    n = int(round((b - a) / h))
    return h * sum(f(a + (i + 0.5) * h) for i in range(n))


def integrate_trapezoid(f, a, b, h):
    """Метод трапеций (p = 2)"""

    n = int(round((b - a) / h))
    s = 0.5 * (f(a) + f(b)) + sum(f(a + i * h) for i in range(1, n))
    return h * s


def integrate_simpson(f, a, b, h):
    """Метод Симпсона (парабол) (p = 4)"""

    n = int(round((b - a) / h))
    if n % 2 != 0:
        raise ValueError(f"Для метода Симпсона число разбиений n = {n} должно быть четным!")
    
    s = f(a) + f(b)
    for i in range(1, n):
        weight = 4.0 if i % 2 == 1 else 2.0
        s += weight * f(a + i * h)
        
    return (h / 3.0) * s


def runge_romberg_refinement(F_h1, F_h2, h1, h2, p):
    """Уточнение значения интеграла методом Рунге — Ромберга — Ричардсона"""

    k = h1 / h2
    error_est = (F_h2 - F_h1) / (k ** p - 1.0)
    F_refined = F_h2 + error_est
    return F_refined, abs(error_est)


def theoretical_error_bounds(task, h):
    """Оценка погрешности методов интегрирования по мажорантным формулам"""

    a, b = task.x0, task.x1
    grid = np.linspace(a, b, 2000)
    
    M1 = max(abs(task.d1f(x)) for x in grid)
    M2 = max(abs(task.d2f(x)) for x in grid)
    M4 = max(abs(task.d4f(x)) for x in grid)
    
    R_rect_lr = ((b - a) / 2.0) * h * M1
    
    R_rect_mid = ((b - a) / 24.0) * (h ** 2) * M2
    
    R_trap = ((b - a) / 12.0) * (h ** 2) * M2

    R_simp = ((b - a) / 180.0) * (h ** 4) * M4
    
    return {
        'M1': M1, 'M2': M2, 'M4': M4,
        'left': R_rect_lr,
        'right': R_rect_lr,
        'mid': R_rect_mid,
        'trap': R_trap,
        'simp': R_simp
    }


def main():
    task = NumericalIntegrationTask()
    exact_val = task.exact_integral()
    
    print("=" * 112)
    print(f" ЧИСЛЕННОЕ ИНТЕГРИРОВАНИЕ")
    print("=" * 112)
    print(f"  Подынтегральная функция : y = x^2 / (x^3 - 27)")
    print(f"  Отрезок интегрирования  : [{task.x0}; {task.x1}]")
    print(f"  Шаги интегрирования     : h1 = {task.h1}, h2 = {task.h2}")
    print(f"  Точное значение F_exact : {exact_val:.8f}")
    
    # Список всех методов по требованиям лабораторной работы
    methods = [
        ("1. Левые прямоугольники",   integrate_rect_left,  1, 'left'),
        ("2. Правые прямоугольники",  integrate_rect_right, 1, 'right'),
        ("3. Средние прямоугольники", integrate_rect_mid,   2, 'mid'),
        ("4. Метод трапеций",         integrate_trapezoid,  2, 'trap'),
        ("5. Метод Симпсона",         integrate_simpson,    4, 'simp')
    ]
    
    results = []
    for name, func, p, key in methods:
        F_h1 = func(task.f, task.x0, task.x1, task.h1)
        F_h2 = func(task.f, task.x0, task.x1, task.h2)
        F_rr, err_rr = runge_romberg_refinement(F_h1, F_h2, task.h1, task.h2, p)
        
        results.append({
            'name': name,
            'key': key,
            'p': p,
            'F_h1': F_h1,
            'F_h2': F_h2,
            'F_rr': F_rr,
            'err_rr': err_rr,
            'err_h1': abs(exact_val - F_h1),
            'err_h2': abs(exact_val - F_h2),
            'err_rr_exact': abs(exact_val - F_rr)
        })

    # Таблица 1: Результаты интегрирования и уточнение Рунге-Ромберга
    print("\n" + "-" * 112)
    print("1. СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ ИНТЕГРИРОВАНИЯ И УТОЧНЕНИЯ МЕТОДОМ РУНГЕ — РОМБЕРГА")
    print("-" * 112)
    header = (
        f"{'Метод интегрирования':<27} | {'Порядок p':<9} | "
        f"{f'Шаг h1 = {task.h1}':<14} | {f'Шаг h2 = {task.h2}':<14} | "
        f"{'Рунге-Ромберг F*':<16} | {'Погр. |F_ex - F*|':<17}"
    )
    print(header)
    print("-" * 112)
    for r in results:
        print(
            f"{r['name']:<27} | {r['p']:<9} | "
            f"{r['F_h1']:<14.8f} | {r['F_h2']:<14.8f} | "
            f"{r['F_rr']:<16.8f} | {r['err_rr_exact']:<17.3e}"
        )
    print("-" * 112)

    bounds_h1 = theoretical_error_bounds(task, task.h1)
    bounds_h2 = theoretical_error_bounds(task, task.h2)
    
    print("\n" + "-" * 112)
    print("2. ОЦЕНКА ПОГРЕШНОСТИ МЕТОДОВ ИНТЕГРИРОВАНИЯ")
    print("-" * 112)
    print(
        f"  Максимумы модулей производных на [{task.x0}; {task.x1}]: "
        f"M1 = {bounds_h1['M1']:.6f},  M2 = {bounds_h1['M2']:.6f},  M4 = {bounds_h1['M4']:.6f}"
    )
    print("-" * 112)
    header_err = (
        f"{'Метод интегрирования':<27} | {'Факт. погр. (h1)':<16} | {'Оценка R (h1)':<15} | "
        f"{'Факт. погр. (h2)':<16} | {'Оценка R (h2)':<15} | {'Оценка Рунге':<14}"
    )
    print(header_err)
    print("-" * 112)
    for r in results:
        k = r['key']
        print(
            f"{r['name']:<27} | {r['err_h1']:<16.3e} | {bounds_h1[k]:<15.3e} | "
            f"{r['err_h2']:<16.3e} | {bounds_h2[k]:<15.3e} | {r['err_rr']:<14.3e}"
        )
    print("=" * 112)
    print("[ВЫВОД]: Для всех методов фактическая погрешность строго меньше теоретической мажорантной оценки R.")
    print("Наивысшую точность обеспечивает метод Симпсона с уточнением по Рунге — Ромбергу.")


if __name__ == '__main__':
    main()

 ЧИСЛЕННОЕ ИНТЕГРИРОВАНИЕ
  Подынтегральная функция : y = x^2 / (x^3 - 27)
  Отрезок интегрирования  : [-2.0; 2.0]
  Шаги интегрирования     : h1 = 1.0, h2 = 0.5
  Точное значение F_exact : -0.20363636

----------------------------------------------------------------------------------------------------------------
1. СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ ИНТЕГРИРОВАНИЯ И УТОЧНЕНИЯ МЕТОДОМ РУНГЕ — РОМБЕРГА
----------------------------------------------------------------------------------------------------------------
Метод интегрирования        | Порядок p | Шаг h1 = 1.0   | Шаг h2 = 0.5   | Рунге-Ромберг F* | Погр. |F_ex - F*|
----------------------------------------------------------------------------------------------------------------
1. Левые прямоугольники     | 1         | -0.18846154    | -0.18814631    | -0.18783108      | 1.581e-02        
2. Правые прямоугольники    | 1         | -0.28470214    | -0.23626661    | -0.18783108      | 1.581e-02        
3. Средние прямоугольники   | 2     